# GLSKF 视频实验

In [ ]:
import numpy as np
from scipy.sparse import eye, csr_matrix
from scipy.linalg import inv, khatri_rao
import matplotlib.pyplot as plt
import os
import pandas as pd
import time

# 绘图字体设置
# 中文优先使用宋体，英文和数字可回退到 Times New Roman
plt.rcParams['font.sans-serif'] = ['SimSun', 'Times New Roman']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False

def cov_matern(d, loghyper, x):
    ell = np.exp(loghyper[0])
    sf2 = np.exp(2 * loghyper[1])

    def f(t):
        if d == 1:
            return 1
        if d == 3:
            return 1 + t
        if d == 5:
            return 1 + t * (1 + t / 3)
        if d == 7:
            return 1 + t * (1 + t * (6 + t) / 15)
        raise ValueError('不支持的 Matern 阶数')

    def m(t):
        return f(t) * np.exp(-t)

    dist_sq = ((x[:, None] - x[None, :]) / ell) ** 2
    return sf2 * m(np.sqrt(d * dist_sq))

def bohman(loghyper, x):
    range_ = np.exp(loghyper[0])
    dis = np.abs(x[:, None] - x[None, :])
    r = np.minimum(dis / range_, 1)
    k = (1 - r) * np.cos(np.pi * r) + np.sin(np.pi * r) / np.pi
    k[k < 1e-16] = 0
    k[np.isnan(k)] = 0
    return k

def unfold(tensor, mode):
    return np.reshape(np.moveaxis(tensor, mode, 0), (tensor.shape[mode], -1), order='F')

def fold(mat, dim, mode):
    index = [mode]
    for i in range(dim.shape[0]):
        if i != mode:
            index.append(i)
    return np.moveaxis(np.reshape(mat, list(dim[index]), order='F'), 0, mode)

def kroneckerMVM(K3, K2, K1, vec, d1, d2, d3):
    temp1 = (K1 @ vec.reshape(d1, d2, d3, order='F').reshape(d1, -1)).reshape(d1, d2, d3)
    temp2 = (K2 @ temp1.transpose(1, 0, 2).reshape(d2, -1)).reshape(d2, d1, d3).transpose(1, 0, 2)
    temp3 = (K3 @ temp2.transpose(2, 0, 1).reshape(d3, -1)).reshape(d3, d1, d2).transpose(1, 2, 0)
    return temp3.ravel(order='F')

def Ap_operatorT(vec, maskT, KrU, KrU_T, Qu, rho, rank_num, mode_dim):
    x = vec.reshape(rank_num, mode_dim, order='F')
    temp = KrU @ x
    temp *= maskT
    ap1 = KrU_T @ temp
    ap2 = rho * (x @ Qu)
    return (ap1 + ap2).ravel(order='F')

def cg_factorT(Qu, rho, KrU, mask_matrixT, y_tilde, priorvalue, max_iter):
    rank_num, mode_dim = y_tilde.shape
    y_flat = y_tilde.ravel(order='F')
    x = priorvalue.copy()
    KrU_T = KrU.T
    ax = Ap_operatorT(x, mask_matrixT, KrU, KrU_T, Qu, rho, rank_num, mode_dim)
    r = y_flat - ax
    p = r.copy()
    rsold = np.dot(r, r)
    approxE = np.zeros(max_iter)

    for i in range(max_iter):
        ap = Ap_operatorT(p, mask_matrixT, KrU, KrU_T, Qu, rho, rank_num, mode_dim)
        alpha = rsold / np.dot(p, ap)
        x += alpha * p
        r -= alpha * ap
        rsnew = np.dot(r, r)
        approxE[i] = np.sqrt(rsnew)
        if approxE[i] < 1e-4:
            break
        p = r + (rsnew / rsold) * p
        rsold = rsnew
    return x, approxE

def Ap_operatorL(vec, pos_obs, Kd, Kt, Ks, gamma, d1, d2, d3):
    x = np.zeros(d1 * d2 * d3)
    x[pos_obs] = vec
    ap1 = kroneckerMVM(Kd, Kt, Ks, x, d1, d2, d3)
    return ap1[pos_obs] + gamma * vec

def cg_local(gamma, Kd, Kt, Ks, pos_obs, y_tilde, priorvalue, max_iter):
    d1, d2, d3 = y_tilde.shape
    y_obs = y_tilde.ravel(order='F')[pos_obs]
    x = priorvalue.copy()
    ax = Ap_operatorL(x, pos_obs, Kd, Kt, Ks, gamma, d1, d2, d3)
    r = y_obs - ax
    p = r.copy()
    rsold = np.dot(r, r)
    approxE = np.zeros(max_iter)

    for i in range(max_iter):
        ap = Ap_operatorL(p, pos_obs, Kd, Kt, Ks, gamma, d1, d2, d3)
        alpha = rsold / np.dot(p, ap)
        x += alpha * p
        r -= alpha * ap
        rsnew = np.dot(r, r)
        approxE[i] = np.sqrt(rsnew)
        if approxE[i] < 1e-4:
            break
        p = r + (rsnew / rsold) * p
        rsold = rsnew
    return x, approxE

def GLSKF(I, Omega, lengthscaleU, lengthscaleR, varianceU, varianceR, tapering_range, d_MaternU, d_MaternR, rank_num, rho, gamma, maxiter, K0, epsilon):
    N = np.array(I.shape)
    D = I.ndim
    maxP = 255.0

    Omega = Omega.astype(bool)
    pos_miss = np.where(~Omega)
    num_obser = np.sum(Omega)
    mask_matrix = [unfold(Omega, d) for d in range(D)]
    mask_matrixT = [mask_matrix[d].T for d in range(D)]
    pos_obs = np.where(mask_matrix[0].ravel(order='F') == 1)[0]
    idx_frame = np.sum(mask_matrix[2], axis=0) > 0

    train_values = I[Omega]
    train_mean = float(np.mean(train_values))
    Isubmean = I.astype(float) - train_mean
    T = Isubmean * Omega

    hyper_Ku = [None] * D
    hyper_Ku[0] = [np.log(lengthscaleU[0]), np.log(varianceU[0])]
    hyper_Ku[1] = [np.log(lengthscaleU[1]), np.log(varianceU[1])]

    hyper_Kr = [None] * D
    hyper_Kr[0] = [np.log(lengthscaleR[0]), np.log(varianceR[0]), np.log(tapering_range)]
    hyper_Kr[1] = [np.log(lengthscaleR[1]), np.log(varianceR[1]), np.log(tapering_range)]

    invKu = [None] * D
    Kr = [None] * D

    x = np.arange(1, N[0] + 1)
    Ku1 = cov_matern(d_MaternU, hyper_Ku[0], x)
    invKu[0] = inv(Ku1)
    taper1 = bohman([hyper_Kr[0][2]], x)
    Kr[0] = csr_matrix(cov_matern(d_MaternR, hyper_Kr[0][:2], x) * taper1)

    x = np.arange(1, N[1] + 1)
    Ku2 = cov_matern(d_MaternU, hyper_Ku[1], x)
    invKu[1] = inv(Ku2)
    taper2 = bohman([hyper_Kr[1][2]], x)
    Kr[1] = csr_matrix(cov_matern(d_MaternR, hyper_Kr[1][:2], x) * taper2)

    invKu[2] = csr_matrix(eye(N[2], format='csr'))
    Kr[2] = csr_matrix(eye(N[2], format='csr'))

    X = T.copy()
    X[pos_miss] = T.sum() / num_obser

    U = [0.1 * np.random.randn(N[d], rank_num) for d in range(D)]
    M_unfold1 = U[0] @ khatri_rao(U[2], U[1]).T
    M = fold(M_unfold1, N, 0)

    UTvector = [U[d].T.ravel(order='F') for d in range(D)]
    Rtensor = np.zeros(N, dtype=float)
    Rvector_temp = np.zeros(N[0] * N[1] * N[2], dtype=float)
    X[pos_miss] = M[pos_miss] + Rtensor[pos_miss]

    d_all = np.arange(0, D)
    train_norm = np.linalg.norm(T)
    last_tensor = T.copy()
    psnr_hist = np.zeros(maxiter)
    iter_num = 0

    while True:
        Gtensor = X - Rtensor
        Gtensor_mask = Gtensor * Omega

        for d in range(D):
            dsub = np.delete(d_all, d)
            KrU = khatri_rao(U[dsub[1]], U[dsub[0]])
            HG = KrU.T @ unfold(Gtensor_mask, d).T
            UTvector[d], _ = cg_factorT(invKu[d], rho, KrU, mask_matrixT[d], HG, UTvector[d], 100)
            U[d] = UTvector[d].reshape(rank_num, N[d], order='F').T

        M_unfold1 = U[0] @ khatri_rao(U[2], U[1]).T
        M = fold(M_unfold1, N, 0)
        X[pos_miss] = M[pos_miss] + Rtensor[pos_miss]

        if iter_num >= K0:
            Ltensor = X - M
            Ltensor_mask = Ltensor * Omega
            Rvector_temp[pos_obs], _ = cg_local(gamma, Kr[2], Kr[1], Kr[0], pos_obs, Ltensor_mask, Rvector_temp[pos_obs], 100)
            Rvector = kroneckerMVM(Kr[2], Kr[1], Kr[0], Rvector_temp, N[0], N[1], N[2])
            Rtensor = Rvector.reshape(N, order='F')
            Rtensor_unfold3 = unfold(Rtensor, 2)
            if np.sum(idx_frame) > 1:
                Rtensor_unfold3_obs = Rtensor_unfold3[:, idx_frame]
                Kr[2] = np.cov(Rtensor_unfold3_obs)
        else:
            Rtensor = np.zeros_like(Rtensor)

        X[pos_miss] = M[pos_miss] + Rtensor[pos_miss]
        Xori = X + train_mean
        Xrecovery = np.maximum(0, Xori)
        Xrecovery = np.minimum(maxP, Xrecovery)

        mse_val = np.mean((I.astype(float) - Xrecovery) ** 2)
        psnr_hist[iter_num] = 10 * np.log10(maxP ** 2 / max(mse_val, np.finfo(float).eps))

        iter_num += 1
        print(f'Epoch = {iter_num}, PSNR = {psnr_hist[iter_num - 1]:.4f} dB')

        tol = np.linalg.norm(X - last_tensor) / train_norm
        last_tensor = X.copy()
        if (tol < epsilon) or (iter_num >= maxiter):
            break

    return Xrecovery, psnr_hist[:iter_num]


In [ ]:
# 参数设置
seedr = 920
np.random.seed(seedr)

# 需要依次运行的视频文件
video_files = [
    {
        'video_name': 'news_qcif',
        'filename': 'news_qcif_gray.yuv'
    },
 {
        'video_name': 'akiyo_qcif',
        'filename': 'akiyo_qcif_gray.yuv'
 }
]

# 自动识别当前运行目录
base_dir = os.getcwd()
candidate_dirs = [
    base_dir,
    os.path.join(base_dir, '视频实验')
]

video_dir = None
for candidate_dir in candidate_dirs:
    processed_dir = os.path.join(candidate_dir, '处理后视频')
    if all(os.path.exists(os.path.join(processed_dir, item['filename'])) for item in video_files):
        video_dir = candidate_dir
        break

if video_dir is None:
    raise FileNotFoundError('没有找到同时包含 news_qcif_gray.yuv 和 akiyo_qcif_gray.yuv 的处理后视频目录')

# 视频格式设置
w = 176
h = 144
fps = 30
show_idx = [40, 200]

# 实验参数
missing_rate = [0.8,0.9,0.95]
lengthscaleU = np.ones(2) * 30
varianceU = np.ones(2)
lengthscaleR = np.ones(2) * 5
varianceR = np.ones(2)
tapering_range = 10
d_MaternU = 3
d_MaternR = 3
rank_num = 10
maxiter = 20
K0 = 10
epsilon = 1e-4

# 论文里的候选范围
rho_list = [1, 5, 10]
gamma_list = [1, 5, 10]

def read_yuv420_gray(video_path, height, width):
    """读取 YUV420 灰度视频，只保留 Y 平面。"""
    y_size = width * height
    uv_size = y_size // 4
    frame_size = y_size + 2 * uv_size
    file_size = os.path.getsize(video_path)

    if file_size % frame_size != 0:
        raise ValueError(f'视频大小与 QCIF YUV420 格式不匹配: {video_path}')

    frame_num = file_size // frame_size
    video = np.zeros((height, width, frame_num), dtype=np.uint8)

    with open(video_path, 'rb') as f:
        for k in range(frame_num):
            y = np.frombuffer(f.read(y_size), dtype=np.uint8)
            if y.size != y_size:
                raise ValueError(f'{video_path} 读取第{k + 1}帧失败')
            video[:, :, k] = y.reshape((height, width))
            f.seek(2 * uv_size, 1)

    return video, frame_num

all_results = []

for video_item in video_files:
    video_name = video_item['video_name']
    video_path = os.path.join(video_dir, '处理后视频', video_item['filename'])
    I, frame_num = read_yuv420_gray(video_path, h, w)

    print('\n' + '#' * 60)
    print(f'开始处理视频: {video_item["filename"]}')
    print(f'视频路径: {video_path}')
    print(f'尺寸: {h} x {w} x {frame_num}')
    print('#' * 60)

    for current_missing_rate in missing_rate:
        # 生成缺失掩码
        np.random.seed(seedr)
        Omega = np.random.rand(h, w, frame_num) > current_missing_rate
        observed = (I.astype(float) * Omega).astype(np.uint8)

        # 输出目录，例如 news_qcif_90、akiyo_qcif_90
        result_dir = os.path.join(
            video_dir,
            'results',
            'GLSKF',
            f'{video_name}_{int(current_missing_rate * 100)}'
        )
        os.makedirs(result_dir, exist_ok=True)

        print('\n' + '=' * 60)
        print('GLSKF 视频实验')
        print('=' * 60)
        print(f'视频: {video_path}')
        print(f'尺寸: {h} x {w} x {frame_num}')
        print(f'帧率: {fps}')
        print(f'缺失率: {current_missing_rate * 100:.1f}%')
        print(f'观测像素: {np.sum(Omega)} / {I.size}')
        print(f'输出目录: {result_dir}')
        print('=' * 60 + '\n')

        best_psnr = -np.inf
        best_params = None
        best_result = None
        summary_rows = []

        for rho in rho_list:
            for gamma in gamma_list:
                print(f'测试参数: video={video_name}, missing_rate={current_missing_rate:.2f}, rho={rho}, gamma={gamma}')
                start_time = time.perf_counter()
                X, psnr_hist = GLSKF(
                    I, Omega, lengthscaleU, lengthscaleR, varianceU, varianceR,
                    tapering_range, d_MaternU, d_MaternR, rank_num, rho, gamma,
                    maxiter, K0, epsilon
                )
                elapsed = time.perf_counter() - start_time

                diff = I.astype(float) - X
                mse_val = np.mean(diff ** 2)
                rmse_val = np.sqrt(mse_val)
                psnr_val = 10 * np.log10(255 ** 2 / max(mse_val, np.finfo(float).eps))
                best_epoch_psnr = float(np.max(psnr_hist))
                best_epoch = int(np.argmax(psnr_hist) + 1)

                summary_rows.append({
                    'Video': video_name,
                    'MissingRate': current_missing_rate,
                    'rho': rho,
                    'gamma': gamma,
                    'PSNR': psnr_val,
                    'MSE': mse_val,
                    'RMSE': rmse_val,
                    'BestEpochPSNR': best_epoch_psnr,
                    'BestEpoch': best_epoch,
                    'Time': elapsed
                })

                print(f'最终 PSNR: {psnr_val:.4f} dB')
                print(f'MSE: {mse_val:.6f}')
                print(f'RMSE: {rmse_val:.6f}')
                print(f'Best Epoch PSNR: {best_epoch_psnr:.4f} dB')
                print(f'Best Epoch: {best_epoch}')
                print(f'耗时: {elapsed:.2f} 秒\n')

                if psnr_val > best_psnr:
                    best_psnr = psnr_val
                    best_params = {'rho': rho, 'gamma': gamma}
                    best_result = {
                        'recovered': X,
                        'psnr_hist': psnr_hist,
                        'psnr': psnr_val,
                        'mse': mse_val,
                        'rmse': rmse_val,
                        'best_epoch_psnr': best_epoch_psnr,
                        'best_epoch': best_epoch,
                        'time': elapsed
                    }

        print('=' * 60)
        print(f'视频: {video_name}')
        print(f'缺失率: {current_missing_rate * 100:.1f}%')
        print(f'最优参数: rho={best_params["rho"]}, gamma={best_params["gamma"]}')
        print(f'最终 PSNR: {best_result["psnr"]:.4f} dB')
        print(f'MSE: {best_result["mse"]:.6f}')
        print(f'RMSE: {best_result["rmse"]:.6f}')
        print(f'Best Epoch PSNR: {best_result["best_epoch_psnr"]:.4f} dB')
        print(f'Best Epoch: {best_result["best_epoch"]}')
        print('=' * 60)

        all_results.append({
            'video_name': video_name,
            'video_path': video_path,
            'I': I,
            'height': h,
            'width': w,
            'frame_num': frame_num,
            'fps': fps,
            'show_idx': show_idx,
            'missing_rate': current_missing_rate,
            'Omega': Omega,
            'observed': observed,
            'result_dir': result_dir,
            'best_params': best_params,
            'best_result': best_result,
            'summary_rows': summary_rows
        })


In [ ]:
# 保存指标和对比图
for item in all_results:
    video_name = item['video_name']
    video_path = item['video_path']
    I = item['I']
    h = item['height']
    w = item['width']
    frame_num = item['frame_num']
    fps = item['fps']
    show_idx = item['show_idx']

    current_missing_rate = item['missing_rate']
    Omega = item['Omega']
    observed = item['observed']
    result_dir = item['result_dir']
    best_params = item['best_params']
    best_result = item['best_result']
    summary_rows = item['summary_rows']

    metrics_path = os.path.join(result_dir, 'metrics.txt')
    with open(metrics_path, 'w', encoding='utf-8') as f:
        f.write(f'video_name={video_name}\n')
        f.write(f'video={video_path}\n')
        f.write(f'size={h} x {w} x {frame_num}\n')
        f.write(f'fps={fps}\n')
        f.write(f'missing_rate={current_missing_rate:.2f}\n')
        f.write(f'rho={best_params["rho"]}\n')
        f.write(f'gamma={best_params["gamma"]}\n')
        f.write(f'psnr={best_result["psnr"]:.6f}\n')
        f.write(f'mse={best_result["mse"]:.6f}\n')
        f.write(f'rmse={best_result["rmse"]:.6f}\n')
        f.write(f'best_epoch_psnr={best_result["best_epoch_psnr"]:.6f}\n')
        f.write(f'best_epoch={best_result["best_epoch"]}\n')
        f.write(f'time={best_result["time"]:.6f}\n')

    np.savez(
        os.path.join(result_dir, 'result.npz'),
        video_name=video_name,
        video_path=video_path,
        I=I,
        Omega=Omega,
        observed=observed,
        recovered=best_result['recovered'],
        psnr_hist=best_result['psnr_hist']
    )

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_excel(os.path.join(result_dir, '实验总结.xlsx'), index=False, engine='openpyxl')

    show_idx_valid = [idx for idx in show_idx if idx <= frame_num]
    for idx in show_idx_valid:
        ori = I[:, :, idx - 1]
        obs = observed[:, :, idx - 1]
        rec = np.uint8(best_result['recovered'][:, :, idx - 1])

        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        plt.imshow(ori, cmap='gray')
        plt.title(f'Original {idx}')
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.imshow(obs, cmap='gray')
        plt.title(f'Observed {idx}')
        plt.axis('off')

        plt.subplot(1, 3, 3)
        plt.imshow(rec, cmap='gray')
        plt.title(f'Recovered {idx}')
        plt.axis('off')

        plt.tight_layout()
        plt.savefig(os.path.join(result_dir, f'frame_{idx:03d}_compare.png'), dpi=150, bbox_inches='tight')
        plt.show()

    plt.figure(figsize=(10, 6))
    plt.plot(np.arange(1, len(best_result['psnr_hist']) + 1), best_result['psnr_hist'], linewidth=2)
    plt.xlabel('迭代次数')
    plt.ylabel('PSNR (dB)')
    plt.title(
        f'GLSKF 视频收敛曲线\n'
        f'{video_name}, missing_rate={current_missing_rate:.2f}, '
        f'rho={best_params["rho"]}, gamma={best_params["gamma"]}'
    )
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(result_dir, '收敛曲线.png'), dpi=150, bbox_inches='tight')
    plt.show()

print('结果已保存到:')
for item in all_results:
    print(item['result_dir'])
